In [1]:
import numpy as np
import json
import matplotlib.pyplot as plt

In [2]:
def prepend_ones(arr):
    ones = np.ones((arr.shape[0], 1))
    return np.hstack((ones, arr))

def load_vision_method_soft(model, method):
    with open(f"../scripts/_cache/soft_rates_{method}_{model}_imagenet_2_per_class.json") as f:
        return prepend_ones(np.array(json.load(f)["soft_stability_rates"]))

def load_vision_method_hard(model, method):
    with open(f"../scripts/_cache/hard_radii_{method}_{model}_imagenet_2_per_class.json") as f:
        l2r = json.load(f)["lambda_to_radii"]
        return {lambda_: np.array(l2r[lambda_]) for lambda_ in ["0.125", "0.250", "0.375", "0.500"]}

methods = ["lime", "shap", "intgrad", "mfaba", "random"]
vit_soft_rates = {m: load_vision_method_soft("vit", m) for m in methods}
resnet50_soft_rates = {m: load_vision_method_soft("resnet50", m) for m in methods}
resnet18_soft_rates = {m: load_vision_method_soft("resnet18", m) for m in methods}

vit_hard_radii = {m: load_vision_method_hard("vit", m) for m in methods}
resnet50_hard_radii = {m: load_vision_method_hard("resnet50", m) for m in methods}
resnet18_hard_radii = {m: load_vision_method_hard("resnet18", m) for m in methods}

In [3]:
def right_last_padded(lists):
    max_len = max(len(ls) for ls in lists)
    padded_array = np.array([ls + [ls[-1]] * (max_len - len(ls)) for ls in lists])
    return padded_array

def load_roberta_task_method_soft(task, method):
    with open(f"../scripts/_cache/soft_rates_{method}_roberta_tweeteval_{task}.json") as f:
        return prepend_ones(right_last_padded(json.load(f)["soft_stability_rates"]))

def load_roberta_task_method_hard(task, method):
    with open(f"../scripts/_cache/hard_radii_{method}_roberta_tweeteval_{task}.json") as f:
        l2r = json.load(f)["lambda_to_radii"]
        return {lambda_: np.array(l2r[lambda_]) for lambda_ in ["0.125", "0.250", "0.375", "0.500"]}

In [4]:
roberta_emoji_soft_rates = {m: load_roberta_task_method_soft("emoji", m) for m in methods}
roberta_emotion_soft_rates = {m: load_roberta_task_method_soft("emotion", m) for m in methods}
roberta_hate_soft_rates = {m: load_roberta_task_method_soft("hate", m) for m in methods}
roberta_irony_soft_rates = {m: load_roberta_task_method_soft("irony", m) for m in methods}
roberta_offensive_soft_rates = {m: load_roberta_task_method_soft("offensive", m) for m in methods}
roberta_sentiment_soft_rates = {m: load_roberta_task_method_soft("sentiment", m) for m in methods}

roberta_soft_rates = {
    m: np.vstack([
        roberta_emoji_soft_rates[m],
        roberta_emotion_soft_rates[m],
        roberta_hate_soft_rates[m],
        roberta_irony_soft_rates[m],
        roberta_offensive_soft_rates[m],
        roberta_sentiment_soft_rates[m]
    ])
    for m in methods
}

roberta_emoji_hard_radii = {m: load_roberta_task_method_hard("emoji", m) for m in methods}
roberta_emotion_hard_radii = {m: load_roberta_task_method_hard("emotion", m) for m in methods}
roberta_hate_hard_radii = {m: load_roberta_task_method_hard("hate", m) for m in methods}
roberta_irony_hard_radii = {m: load_roberta_task_method_hard("irony", m) for m in methods}
roberta_offensive_hard_radii = {m: load_roberta_task_method_hard("offensive", m) for m in methods}
roberta_sentiment_hard_radii = {m: load_roberta_task_method_hard("sentiment", m) for m in methods}

roberta_hard_radii = {
    m : {
        lambda_: np.concatenate([
            roberta_emoji_hard_radii[m][lambda_],
            roberta_emotion_hard_radii[m][lambda_],
            roberta_hate_hard_radii[m][lambda_],
            roberta_irony_hard_radii[m][lambda_],
            roberta_offensive_hard_radii[m][lambda_],
            roberta_sentiment_hard_radii[m][lambda_],
        ])
        for lambda_ in ["0.125", "0.250", "0.375", "0.500"]
    }
    for m in methods
}

In [5]:
vit_soft_dict = {
    "radii": list(range(0,21)) + [30, 40, 50, 60, 70, 80, 90, 100],
    "lime": vit_soft_rates["lime"].mean(axis=0).tolist(),
    "shap": vit_soft_rates["shap"].mean(axis=0).tolist(),
    "intgrad": vit_soft_rates["intgrad"].mean(axis=0).tolist(),
    "mfaba": vit_soft_rates["mfaba"].mean(axis=0).tolist(),
    "random": vit_soft_rates["random"].mean(axis=0).tolist(),
}

with open("_dump/json/blog_vit_soft_stability.json", "w") as f:
    f.write(json.dumps(vit_soft_dict, indent=4))

In [6]:
roberta_soft_dict = {
    "radii": list(range(0,21)),
    "lime": roberta_soft_rates["lime"].mean(axis=0).tolist(),
    "shap": roberta_soft_rates["shap"].mean(axis=0).tolist(),
    "intgrad": roberta_soft_rates["intgrad"].mean(axis=0).tolist(),
    "mfaba": roberta_soft_rates["mfaba"].mean(axis=0).tolist(),
    "random": roberta_soft_rates["random"].mean(axis=0).tolist(),
}

with open("_dump/json/blog_roberta_soft_stability.json", "w") as f:
    f.write(json.dumps(roberta_soft_dict, indent=4))

In [7]:
hard_radii = np.linspace(0.0, 2.0, num=100)

vit_hard_dict = {
    "radii": hard_radii.tolist(),
    "lime": [(vit_hard_radii["lime"]["0.250"] >= r).mean() for r in hard_radii],
    "shap": [(vit_hard_radii["shap"]["0.250"] >= r).mean() for r in hard_radii],
    "intgrad": [(vit_hard_radii["intgrad"]["0.250"] >= r).mean() for r in hard_radii],
    "mfaba": [(vit_hard_radii["mfaba"]["0.250"] >= r).mean() for r in hard_radii],
    "random": [(vit_hard_radii["random"]["0.250"] >= r).mean() for r in hard_radii],
}

with open("_dump/json/blog_vit_hard_stability.json", "w") as f:
    f.write(json.dumps(vit_hard_dict, indent=4))

In [8]:
roberta_hard_dict = {
    "radii": hard_radii.tolist(),
    "lime": [(roberta_hard_radii["lime"]["0.250"] >= r).mean() for r in hard_radii],
    "shap": [(roberta_hard_radii["shap"]["0.250"] >= r).mean() for r in hard_radii],
    "intgrad": [(roberta_hard_radii["intgrad"]["0.250"] >= r).mean() for r in hard_radii],
    "mfaba": [(roberta_hard_radii["mfaba"]["0.250"] >= r).mean() for r in hard_radii],
    "random": [(roberta_hard_radii["random"]["0.250"] >= r).mean() for r in hard_radii],
}

with open("_dump/json/blog_roberta_hard_stability.json", "w") as f:
    f.write(json.dumps(roberta_hard_dict, indent=4))

In [9]:
with open(f"../scripts/_cache/stability_vs_smoothing_resnet50_imagenet_2_per_class.json") as f:
    resnet50_smooth = json.load(f)

with open(f"../scripts/_cache/stability_vs_smoothing_vit_imagenet_2_per_class.json") as f:
    vit_smooth = json.load(f)

In [10]:
resnet50_stability_dict = {
    "radii": list(range(1,21)),
    "lambda_1.0": np.array(resnet50_smooth["lambda_to_rates"]["1.000"])[:,:20].mean(axis=0).tolist(),
    "lambda_0.9": np.array(resnet50_smooth["lambda_to_rates"]["0.900"])[:,:20].mean(axis=0).tolist(),
    "lambda_0.8": np.array(resnet50_smooth["lambda_to_rates"]["0.800"])[:,:20].mean(axis=0).tolist(),
    "lambda_0.7": np.array(resnet50_smooth["lambda_to_rates"]["0.700"])[:,:20].mean(axis=0).tolist(),
    "lambda_0.6": np.array(resnet50_smooth["lambda_to_rates"]["0.600"])[:,:20].mean(axis=0).tolist(),
    "lambda_0.5": np.array(resnet50_smooth["lambda_to_rates"]["0.500"])[:,:20].mean(axis=0).tolist(),
}

with open("_dump/json/blog_resnet50_stability_vs_lambda.json", "w") as f:
    f.write(json.dumps(resnet50_stability_dict, indent=4))

In [11]:
vit_stability_dict = {
    "radii": list(range(1,21)),
    "lambda_1.0": np.array(vit_smooth["lambda_to_rates"]["1.000"])[:,:20].mean(axis=0).tolist(),
    "lambda_0.9": np.array(vit_smooth["lambda_to_rates"]["0.900"])[:,:20].mean(axis=0).tolist(),
    "lambda_0.8": np.array(vit_smooth["lambda_to_rates"]["0.800"])[:,:20].mean(axis=0).tolist(),
    "lambda_0.7": np.array(vit_smooth["lambda_to_rates"]["0.700"])[:,:20].mean(axis=0).tolist(),
    "lambda_0.6": np.array(vit_smooth["lambda_to_rates"]["0.600"])[:,:20].mean(axis=0).tolist(),
    "lambda_0.5": np.array(vit_smooth["lambda_to_rates"]["0.500"])[:,:20].mean(axis=0).tolist(),
}

with open("_dump/json/blog_vit_stability_vs_lambda.json", "w") as f:
    f.write(json.dumps(vit_stability_dict, indent=4))